# AWS Agentcore Stock Assistant Demo

This notebook authenticates against the deployed Cognito user pool, invokes the protected `POST /query` endpoint for the required assessment prompts, and surfaces the streamed API responses plus Langfuse trace metadata captured in the opening `metadata` event.

Before running the notebook:
- Deploy the Terraform stack described in `assestment/README.md`.
- Enable Langfuse in the backend so the `trace_id` and `trace_url` fields resolve to Langfuse Cloud for each streamed run.


In [1]:
import json
import os
import subprocess
from typing import Any, Dict, List
from urllib import error, request

try:
    from IPython.display import JSON, Markdown, display
except ImportError:
    JSON = None
    Markdown = None
    display = print


In [2]:
def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


AWS_REGION = "us-west-1"
COGNITO_USER_POOL_CLIENT_ID = "1sqtaeo6s5se3f7f374klgvorn"
AGENT_QUERY_URL = "https://9te8s835b8.execute-api.us-west-1.amazonaws.com/query"
AGENT_QUERY_METHOD = "POST"
EVALUATOR_EMAIL = "dvizcaya91@gmail.com"
EVALUATOR_PASSWORD = "NuevaContra12$34#"

print(
    json.dumps(
        {
            "aws_region": AWS_REGION,
            "cognito_user_pool_client_id": COGNITO_USER_POOL_CLIENT_ID,
            "agent_query_url": AGENT_QUERY_URL,
            "agent_query_method": AGENT_QUERY_METHOD,
            "evaluator_email": EVALUATOR_EMAIL,
        },
        indent=2,
    )
)


{
  "aws_region": "us-west-1",
  "cognito_user_pool_client_id": "1sqtaeo6s5se3f7f374klgvorn",
  "agent_query_url": "https://9te8s835b8.execute-api.us-west-1.amazonaws.com/query",
  "agent_query_method": "POST",
  "evaluator_email": "dvizcaya91@gmail.com"
}


## Authenticating user with Cognito

In [13]:
def fetch_access_token() -> str:
    command = [
        "aws",
        "cognito-idp",
        "initiate-auth",
        "--region",
        AWS_REGION,
        "--client-id",
        COGNITO_USER_POOL_CLIENT_ID,
        "--auth-flow",
        "USER_PASSWORD_AUTH",
        "--auth-parameters",
        f"USERNAME={EVALUATOR_EMAIL},PASSWORD={EVALUATOR_PASSWORD}",
        "--query",
        "AuthenticationResult.AccessToken",
        "--output",
        "text",
    ]
    print(' '.join(command))
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    access_token = result.stdout.strip()
    if not access_token:
        raise RuntimeError("AWS CLI initiate-auth did not return an access token.")
    return access_token


ACCESS_TOKEN = fetch_access_token()
print(f"Fetched Cognito access token for {EVALUATOR_EMAIL} ({len(ACCESS_TOKEN)} characters).")


aws cognito-idp initiate-auth --region us-west-1 --client-id 1sqtaeo6s5se3f7f374klgvorn --auth-flow USER_PASSWORD_AUTH --auth-parameters USERNAME=dvizcaya91@gmail.com,PASSWORD=NuevaContra12$34# --query AuthenticationResult.AccessToken --output text
Fetched Cognito access token for dvizcaya91@gmail.com (1074 characters).


## Asking questions to the Agent

In [14]:
REQUIRED_QUERIES = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports",
    "I'm researching AMZN give me the current price and any relevant information about their AI business",
    "What is the total amount of office space Amazon owned in North America in 2024?",
]


def invoke_query(query: str, access_token: str) -> List[Dict[str, Any]]:
    payload = json.dumps({"query": query}).encode("utf-8")
    query_request = request.Request(
        AGENT_QUERY_URL,
        data=payload,
        method=AGENT_QUERY_METHOD,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
            "Accept": "application/x-ndjson",
        },
    )

    try:
        with request.urlopen(query_request) as response:
            content_type = response.headers.get("Content-Type", "")
            if "application/x-ndjson" not in content_type:
                raise RuntimeError(f"Unexpected content type: {content_type}")

            events: List[Dict[str, Any]] = []
            for raw_line in response:
                line = raw_line.decode("utf-8").strip()
                if line:
                    events.append(json.loads(line))
            return events
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Endpoint request failed with HTTP {exc.code}: {body}"
        ) from exc


In [15]:
def summarize_events(query: str, events: List[Dict[str, Any]]) -> Dict[str, Any]:
    metadata_event = next(event for event in events if event.get("event") == "metadata")
    message_event = next(event for event in events if event.get("event") == "message")
    complete_event = next(event for event in events if event.get("event") == "complete")
    trace = metadata_event.get("data", {}).get("trace", {})
    answer_text = message_event.get("data", {}).get("text", "")

    return {
        "query": query,
        "event_count": len(events),
        "trace": trace,
        "metadata_event": metadata_event,
        "message_event": message_event,
        "complete_event": complete_event,
        "answer_preview": answer_text,
    }


def escape_markdown_cell(value: str) -> str:
    return value.replace("|", "\\|").replace("\n", " ")


In [16]:
runs: List[Dict[str, Any]] = []
for query in REQUIRED_QUERIES:
    events = invoke_query(query, ACCESS_TOKEN)
    runs.append({
        "query": query,
        "events": events,
        "summary": summarize_events(query, events),
    })

print(f"Collected {len(runs)} authenticated streamed runs.")


Collected 5 authenticated streamed runs.


In [17]:
summary_lines = [
    "| Query | Trace ID |Answer Preview |",
    "| --- | --- | --- | --- |",
]

for run in runs:
    summary = run["summary"]
    trace = summary["trace"]
    summary_lines.append(
        "| {query} | {trace_id} | {trace_url} | {answer_preview} |".format(
            query=escape_markdown_cell(summary["query"]),
            trace_id=escape_markdown_cell(str(trace.get("trace_id"))),
            answer_preview=escape_markdown_cell(summary["answer_preview"]),
        )
    )

summary_markdown = "\n".join(summary_lines)

print(summary_markdown)


KeyError: 'trace_url'

## Langfuse logs

In [ ]:
for run in runs:
    summary = run["summary"]
    heading = f"### {summary['query']}"
    if Markdown is not None:
        display(Markdown(heading))
    else:
        print(heading)

    print("Langfuse trace metadata:")
    print(json.dumps(summary["trace"], indent=2))
    print("Metadata event:")
    print(json.dumps(summary["metadata_event"], indent=2))
    print("Final message event:")
    print(json.dumps(summary["message_event"], indent=2))
    print("Complete event:")
    print(json.dumps(summary["complete_event"], indent=2))
